### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1.A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)

2.A function or coroutine to execute.

In [5]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import AzureChatOpenAI
os.environ["AZURE_OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_API_KEY")
# 1. Load the environment variables from your .env file
load_dotenv()

# 2. Explicitly initialize the Azure OpenAI model
# Make sure to replace "gpt-4o" with your Azure deployment name if it is named differently in your Azure Portal.
azure_model = AzureChatOpenAI(
    azure_deployment="gpt-4o", 
    api_version="2025-01-01-preview"  # Check your Azure portal for the correct API version if needed
)



In [6]:
from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4o",model_provider="azure_openai",api_version="2025-01-01-preview")
response = model.invoke("why do parrots talk")
response

AIMessage(content='Parrots are known for their remarkable ability to mimic human speech and other sounds in their environment. This talent comes down to their unique anatomy, intelligence, and social nature. Here are the main reasons why parrots "talk":\n\n---\n\n### 1. **Highly Developed Vocal Learning Ability**\nParrots are vocal learners, meaning they can listen to sounds, process them, and reproduce them. This skill is relatively rare in the animal kingdom and is shared by only a few species, including humans, songbirds, and some marine mammals like dolphins. \n\nParrots possess a unique vocal organ called the **syrinx**, located at the base of their trachea, which gives them precise control over the sounds they produce. They don\'t have vocal cords like humans, but their syrinx allows them to reproduce complex sounds with surprising accuracy.\n\n---\n\n### 2. **Intelligence and Problem-Solving Skills**\nParrots are highly intelligent birds, often compared to the cognitive abilitie

In [7]:
from langchain.tools import tool

@tool 
def get_weather(location:str) ->str:
    """ Get the weather at a location"""
    return f"It's sunny in{location}"

model_with_tools=model.bind_tools([get_weather])

In [8]:
response = model_with_tools.invoke("What's the weather like in boston?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call["args"]}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 52, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 43, 'engine_ttlt_ms': 138, 'pre_inference_ms': 77, 'service_tbt_ms': 17, 'service_ttft_ms': 837, 'service_ttlt_ms': 1089, 'total_duration_ms': 1028, 'user_visible_ttft_ms': 760}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_6ef6b02e87', 'id': 'chatcmpl-E52NPtiFuzHFyw3ChBkyON31jG3iA', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': 

### Tool Execution Loops

In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston!


In [10]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 50, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 59, 'engine_ttlt_ms': 139, 'pre_inference_ms': 47, 'service_tbt_ms': 13, 'service_ttft_ms': 607, 'service_ttlt_ms': 686, 'total_duration_ms': 757, 'user_visible_ttft_ms': 560}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_91d870f097', 'id': 'chatcmpl-E52PCkiFOloxsuC9dVkgwPyg1GH5a', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jail